## 面试问题

循环 step 级幂等：同一步被重试重复执行怎么不污染状态？

## 回答主线

这里的幂等是 loop step 级，不是工具调用级。当控制器重试、崩溃恢复或事件重放导致同一 step 被重复应用时，循环状态不能被重复推进。做法是给每个 step 一个确定性 `step_id`，状态里维护 `applied` 集合，`reduce` 前先查该 step 是否已应用。本 Notebook 模拟 s2 被重复投递，对比无幂等（账本翻倍、步数=3）与幂等去重（账本正确、步数=2）。

## 真实案例

退款循环两步 `s1=verify`、`s2=refund`。模拟 s2 在崩溃恢复中被重复投递一次，重放流为 `[s1, s2, s2]`。对比无幂等与带 `applied` 去重的状态推进。数据为教学重放流，不代表真实崩溃恢复实现。

In [1]:
loop_steps = [  # 定义一段将被循环应用的 step 序列。
    {"step_id": "s1", "action": "verify"},  # 第一步校验。
    {"step_id": "s2", "action": "refund"},  # 第二步退款。
]  # 结束 step 序列定义。

replay_stream = [loop_steps[0], loop_steps[1], loop_steps[1]]  # 模拟 s2 因恢复被重复投递一次。
print("应生效步数:", len(loop_steps))  # 展示原始应生效步数。
print("重放流长度:", len(replay_stream))  # 展示含重复的实际投递流长度。
print("重放流 step_id:", [s["step_id"] for s in replay_stream])  # 展示重放流中出现的重复。

应生效步数: 2
重放流长度: 3
重放流 step_id: ['s1', 's2', 's2']


## 基线（Baseline）

无幂等基线：`reduce` 每次被调用都无条件推进步数并追加账本。正常无重复时它工作正常，但它把「是否重复」的责任完全交给了外部。

In [2]:
def reduce_naive(state, step):  # 无幂等状态推进：每次调用都推进并追加账本。
    new_ledger = state["ledger"] + [step["step_id"]]  # 无条件追加账本条目。
    return {"step": state["step"] + 1, "ledger": new_ledger}  # 无条件推进步数。

## 失败案例与修正

崩溃恢复导致 s2 被重放两次时，无幂等 `reduce` 把它计两遍：账本变成 `[s1,s2,s2]`、步数=3，之后所有基于步数/历史的判断（预算、打转、完成）都被污染。修正是带 `applied` 集合的幂等 `reduce`：同一 `step_id` 只生效一次。

In [3]:
naive_state = {"step": 0, "ledger": []}  # 初始化无幂等状态。
for step in replay_stream:  # 按含重复的重放流逐个应用。
    naive_state = reduce_naive(naive_state, step)  # 无幂等推进导致重复也被应用。
    print("  应用", step["step_id"], "后账本:", naive_state["ledger"])  # 逐步打印无幂等账本演化。
print("无幂等最终步数:", naive_state["step"], "| 账本:", naive_state["ledger"])  # 展示 s2 被重复计入。

  应用 s1 后账本: ['s1']
  应用 s2 后账本: ['s1', 's2']
  应用 s2 后账本: ['s1', 's2', 's2']
无幂等最终步数: 3 | 账本: ['s1', 's2', 's2']


In [4]:
def reduce_idempotent(state, step):  # 幂等状态推进：按 step_id 去重。
    if step["step_id"] in state["applied"]:  # 该 step 已应用过则跳过。
        return state  # 幂等返回不变状态。
    new_applied = state["applied"] | {step["step_id"]}  # 记录已应用的 step_id。
    new_ledger = state["ledger"] + [step["step_id"]]  # 仅首次追加账本。
    return {"step": state["step"] + 1, "ledger": new_ledger, "applied": new_applied}  # 仅首次推进步数。

idem_state = {"step": 0, "ledger": [], "applied": set()}  # 初始化幂等状态。
for step in replay_stream:  # 按同样含重复的重放流应用。
    idem_state = reduce_idempotent(idem_state, step)  # 幂等推进使重复只生效一次。
    print("  应用", step["step_id"], "后账本:", idem_state["ledger"], "applied:", sorted(idem_state["applied"]))  # 逐步打印幂等账本演化。
print("幂等最终步数:", idem_state["step"], "| 账本:", idem_state["ledger"])  # 展示 s2 只计一次。

  应用 s1 后账本: ['s1'] applied: ['s1']
  应用 s2 后账本: ['s1', 's2'] applied: ['s1', 's2']
  应用 s2 后账本: ['s1', 's2'] applied: ['s1', 's2']
幂等最终步数: 2 | 账本: ['s1', 's2']


## 结果解读

无幂等把重复的 s2 计成两步（step=3，账本含两个 s2）；幂等按唯一 `step_id` 去重，step=2、账本 `[s1,s2]`。关键点：幂等要覆盖所有累加量（步数、账本），且 `step_id` 必须确定性生成，否则重放算出不同 id 就失效。这也是确定性重放（题 27）的前提。

In [5]:
print("无幂等 vs 幂等 步数:", naive_state["step"], idem_state["step"])  # 对比两者步数差异。
print("无幂等账本 s2 次数:", naive_state["ledger"].count("s2"))  # 展示无幂等重复计数。
print("幂等账本 s2 次数:", idem_state["ledger"].count("s2"))  # 展示幂等去重效果。

无幂等 vs 幂等 步数: 3 2
无幂等账本 s2 次数: 2
幂等账本 s2 次数: 1


In [6]:
assert naive_state["step"] == 3  # 无幂等把重复 step 也计入。
assert idem_state["step"] == 2  # 幂等只按唯一 step_id 推进。
assert naive_state["ledger"].count("s2") == 2  # 无幂等账本出现重复条目。
assert idem_state["ledger"].count("s2") == 1  # 幂等账本每个 step 只一次。
assert idem_state["ledger"] == ["s1", "s2"]  # 幂等账本顺序与内容正确。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
